# This notebook enables training and testing of Sherlock.
The procedure is:
- Load train, val, test datasets (should be preprocessed)
- Initialize model using the "pretrained" model or by training one from scratch.
- Evaluate and analyse the model predictions.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# This will be the ID for the retrained model,
#further down predictions can also be made with the original model: "sherlock"
model_id = 'retrained_sherlock'

In [3]:
from ast import literal_eval
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score, classification_report

from sherlock.deploy.model import SherlockModel

## Load datasets for training, validation, testing

In [4]:
data_dir = "../custom_data"

In [5]:
start = datetime.now()
print(f'Started at {start}')

X_train = pd.read_parquet(f'{data_dir}/processed/train.parquet')
y_train = pd.read_parquet(f'{data_dir}/raw/train_labels.parquet').values.flatten()

y_train = np.array([x.lower() for x in y_train])

print(f'Load data (train) process took {datetime.now() - start} seconds.')

Started at 2025-08-08 14:50:41.784810
Load data (train) process took 0:00:00.148933 seconds.


In [6]:
print('Distinct types for columns in the Dataframe (should be all float32):')
print(set(X_train.dtypes))

Distinct types for columns in the Dataframe (should be all float32):
{dtype('float32')}


In [7]:
start = datetime.now()
print(f'Started at {start}')

X_validation = pd.read_parquet(f'{data_dir}/processed/validation.parquet')
y_validation = pd.read_parquet(f'{data_dir}/raw/validation_labels.parquet').values.flatten()

y_validation = np.array([x.lower() for x in y_validation])

print(f'Load data (validation) process took {datetime.now() - start} seconds.')

Started at 2025-08-08 14:50:42.781737
Load data (validation) process took 0:00:00.142350 seconds.


In [8]:
start = datetime.now()
print(f'Started at {start}')

X_test = pd.read_parquet(f'{data_dir}/processed/test.parquet')
y_test = pd.read_parquet(f'{data_dir}/raw/test_labels.parquet').values.flatten()

y_test = np.array([x.lower() for x in y_test])

print(f'Finished at {datetime.now()}, took {datetime.now() - start} seconds')

Started at 2025-08-08 14:50:43.544827
Finished at 2025-08-08 14:50:43.665940, took 0:00:00.121119 seconds


## Initialize the model
Two options:
- Load Sherlock model with pretrained weights
- Fit Sherlock model from scratch

### Option 1: load Sherlock with pretrained weights

In [9]:
start = datetime.now()
print(f'Started at {start}')

model = SherlockModel();
model.initialize_model_from_json(with_weights=True, model_id="sherlock");

print('Initialized model.')
print(f'Finished at {datetime.now()}, took {datetime.now() - start} seconds')

W0808 14:50:45.937581 139852823266560 deprecation.py:506] From /home/omadbek/.conda/envs/sherlock/lib/python3.7/site-packages/tensorflow_core/python/ops/init_ops.py:97: calling Zeros.__init__ (from tensorflow.python.ops.init_ops) with dtype is deprecated and will be removed in a future version.
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
W0808 14:50:45.938923 139852823266560 deprecation.py:506] From /home/omadbek/.conda/envs/sherlock/lib/python3.7/site-packages/tensorflow_core/python/ops/init_ops.py:97: calling Ones.__init__ (from tensorflow.python.ops.init_ops) with dtype is deprecated and will be removed in a future version.
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
W0808 14:50:45.943873 139852823266560 deprecation.py:506] From /home/omadbek/.conda/envs/sherlock/lib/python3.7/site-packages/tensorflow_core/python/ops/init_ops.py:97: call

Started at 2025-08-08 14:50:45.927320
Initialized model.
Finished at 2025-08-08 14:50:46.464662, took 0:00:00.537350 seconds


2025-08-08 14:50:46.283862: I tensorflow/core/platform/cpu_feature_guard.cc:142] Your CPU supports instructions that this TensorFlow binary was not compiled to use: AVX2 FMA
2025-08-08 14:50:46.292851: I tensorflow/core/platform/profile_utils/cpu_utils.cc:94] CPU Frequency: 2799955000 Hz
2025-08-08 14:50:46.297880: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x35ba3380 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2025-08-08 14:50:46.297912: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Host, Default Version


### Option 2: fit Sherlock from scratch (and save for later use)

In [10]:
model_id = "retrained_sherlock"

In [11]:
start = datetime.now()
print(f'Started at {start}')

model = SherlockModel()
# Model will be stored with ID `model_id`
model.fit(X_train, y_train, X_validation, y_validation, model_id=model_id)

print('Trained and saved new model.')
print(f'Finished at {datetime.now()}, took {datetime.now() - start} seconds')

Started at 2025-08-08 14:50:49.843715
Train on 433 samples, validate on 54 samples
Epoch 1/200
433/433 [==============================] - 0s 1ms/sample - loss: 3.0834 - categorical_accuracy: 0.0485 - val_loss: 2.7797 - val_categorical_accuracy: 0.0370
Epoch 2/200
433/433 [==============================] - 0s 141us/sample - loss: 2.7658 - categorical_accuracy: 0.1501 - val_loss: 2.6929 - val_categorical_accuracy: 0.1111
Epoch 3/200
433/433 [==============================] - 0s 115us/sample - loss: 2.5018 - categorical_accuracy: 0.3487 - val_loss: 2.6500 - val_categorical_accuracy: 0.1852
Epoch 4/200
433/433 [==============================] - 0s 129us/sample - loss: 2.2175 - categorical_accuracy: 0.4988 - val_loss: 2.6229 - val_categorical_accuracy: 0.1667
Epoch 5/200
433/433 [==============================] - 0s 112us/sample - loss: 2.1103 - categorical_accuracy: 0.5704 - val_loss: 2.6089 - val_categorical_accuracy: 0.1481
Epoch 6/200
433/433 [==============================] - 0s 94us/s

In [12]:
model.store_weights(model_id=model_id)

### Make prediction

In [13]:
predicted_labels = model.predict(X_test, model_id=model_id)
predicted_labels = np.array([x.lower() for x in predicted_labels])

In [14]:
print(f'prediction count {len(predicted_labels)}, type = {type(predicted_labels)}')

size=len(y_test)

# Should be fully deterministic too.
f1_score(y_test[:size], predicted_labels[:size], average="weighted")

prediction count 55, type = <class 'numpy.ndarray'>


0.9055371900826447

In [15]:
# If using the original model, model_id should be replaced with "sherlock"
#model_id = "sherlock"
classes = np.load(f"../model_files/classes_{model_id}.npy", allow_pickle=True)

report = classification_report(y_test, predicted_labels, output_dict=True)

class_scores = list(filter(lambda x: isinstance(x, tuple) and isinstance(x[1], dict) and 'f1-score' in x[1] and x[0] in classes, list(report.items())))

class_scores = sorted(class_scores, key=lambda item: item[1]['f1-score'], reverse=True)

### Top 5 Types

In [16]:
print(f"\t\tf1-score\tprecision\trecall\t\tsupport")

for key, value in class_scores[0:5]:
    if len(key) >= 8:
        tabs = '\t' * 1
    else:
        tabs = '\t' * 2

    print(f"{key}{tabs}{value['f1-score']:.3f}\t\t{value['precision']:.3f}\t\t{value['recall']:.3f}\t\t{value['support']}")

		f1-score	precision	recall		support
age		1.000		1.000		1.000		3
date		1.000		1.000		1.000		12
gender		1.000		1.000		1.000		2
medical_boolean	1.000		1.000		1.000		20
location	0.909		1.000		0.833		6


### Bottom 5 Types

In [17]:
print(f"\t\tf1-score\tprecision\trecall\t\tsupport")

for key, value in class_scores[len(class_scores)-5:len(class_scores)]:
    if len(key) >= 8:
        tabs = '\t' * 1
    else:
        tabs = '\t' * 2

    print(f"{key}{tabs}{value['f1-score']:.3f}\t\t{value['precision']:.3f}\t\t{value['recall']:.3f}\t\t{value['support']}")

		f1-score	precision	recall		support
id		0.750		1.000		0.600		5
case_status	0.667		1.000		0.500		2
contact_setting	0.667		0.500		1.000		1
occupation	0.000		0.000		0.000		1
symptoms	0.000		0.000		0.000		1


### All Scores

In [18]:
print(classification_report(y_test, predicted_labels, digits=3))

                 precision    recall  f1-score   support

            age      1.000     1.000     1.000         3
    case_status      1.000     0.500     0.667         2
contact_setting      0.500     1.000     0.667         1
           date      1.000     1.000     1.000        12
         gender      1.000     1.000     1.000         2
             id      1.000     0.600     0.750         5
       location      1.000     0.833     0.909         6
medical_boolean      1.000     1.000     1.000        20
     occupation      0.000     0.000     0.000         1
        outcome      0.667     1.000     0.800         2
       symptoms      0.000     0.000     0.000         1

       accuracy                          0.891        55
      macro avg      0.742     0.721     0.708        55
   weighted avg      0.942     0.891     0.906        55



## Review errors

In [19]:
size = len(y_test)
mismatches = list()

for idx, k1 in enumerate(y_test[:size]):
    k2 = predicted_labels[idx]

    if k1 != k2:
        mismatches.append(k1)
        
        # zoom in to specific errors. Use the index in the next step
        if k1 in ('address'):
            print(f'[{idx}] expected "{k1}" but predicted "{k2}"')
        
f1 = f1_score(y_test[:size], predicted_labels[:size], average="weighted")
print(f'Total mismatches: {len(mismatches)} (F1 score: {f1})')

data = Counter(mismatches)
data.most_common()   # Returns all unique items and their counts

Total mismatches: 5 (F1 score: 0.8942424242424242)


[('id', 2), ('case_status', 1), ('symptoms', 1), ('occupation', 1)]

In [20]:
test_samples = pd.read_parquet(f'{data_dir}/raw/test_values.parquet')

FileNotFoundError: ../custom_data/raw/test_values.parquet

In [21]:
idx = 1001
original = test_samples.iloc[idx]
converted = original.apply(literal_eval).to_list()

print(f'Predicted "{predicted_labels[idx]}", actual label "{y_test[idx]}". Actual values:\n{converted}')

NameError: name 'test_samples' is not defined